In [0]:
# ================================
# INGESTION PIPELINE — AUTO_FACTORY_2025
# ================================

# carrega funções do notebook utils

In [0]:
import time

In [0]:
%run /Users/brilhance365@gmail.com/AUTO_FACTORY_2025/utils

In [0]:
# ================================
# CONFIGURAÇÃO
# ================================

CATALOG = "workspace"
RAW_SCHEMA = "qap_2025_raw"
BRONZE_SCHEMA = "qap_2025_bronze"
VOLUME = "volume"

BASE_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{VOLUME}"
RAW_PATH = f"{BASE_PATH}/raw_files"
ARCHIVE_PATH = f"{BASE_PATH}/archive"

print("BASE_PATH   :", BASE_PATH)
print("RAW_PATH    :", RAW_PATH)
print("ARCHIVE_PATH:", ARCHIVE_PATH)

In [0]:
# ================================
# LISTAR ARQUIVOS DISPONÍVEIS
# ================================
files = [f for f in dbutils.fs.ls(RAW_PATH) if not f.isDir()]

print("Arquivos encontrados em raw_files:")
for f in files:
    print(f" - {f.name} | {f.path} | {f.size} bytes")

In [0]:
processed_files = []
failed_files = []

for file in files:
    file_path = file.path
    file_name = file.name

    print(f"\n>>> Processando: {file_name}")

    try:
        # 1. leitura do arquivo bruto
        df = read_file_by_extension(spark, file_path)

        # 2. adiciona metadados de ingestão
        df = add_ingestion_metadata(df, file_name)

        # 3. nome da tabela bronze
        table_name = normalize_table_name(file_name)
        full_table_name = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"

        # 4. conta antes de mover o arquivo
        row_count = df.count()

        # 5. grava no bronze
        df.write.format("delta").mode("overwrite").saveAsTable(full_table_name)

        # 6. move para archive somente no final
        archive_file_path = f"{ARCHIVE_PATH}/{file_name}"
        move_to_archive(file_path, archive_file_path)

        processed_files.append({
            "file_name": file_name,
            "table_name": full_table_name,
            "rows": row_count,
            "status": "SUCCESS"
        })

        print(f"✔ Tabela criada : {full_table_name}")
        print(f"✔ Linhas        : {row_count}")
        print(f"✔ Movido p/ arch: {archive_file_path}")

    except Exception as e:
        failed_files.append({
            "file_name": file_name,
            "error": str(e)
        })

        print(f"✖ Erro ao processar: {file_name}")
        print(str(e))

In [0]:
print("\n==============================")
print("RESUMO DA INGESTÃO")
print("==============================")

print(f"\nArquivos processados com sucesso: {len(processed_files)}")
for item in processed_files:
    print(f" - {item['file_name']} -> {item['table_name']} | rows={item['rows']}")

print(f"\nArquivos com erro: {len(failed_files)}")
for item in failed_files:
    print(f" - {item['file_name']} | erro={item['error']}")